# Link Prediction Results Analysis

This notebook analyzes the output of the GraphRAG link prediction experiments and prepares tables and figures for the final report.

Main goals:
- load saved prediction and metric files
- compare methods using Precision@K
- inspect performance across source nodes
- visualize how positive links appear across the ranked list
- extract report-ready conclusions

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use("ggplot")
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current notebook location.")

PROJECT_ROOT = find_project_root(Path.cwd())
RESULTS_DIR = PROJECT_ROOT / "results"
METRICS_PATH = RESULTS_DIR / "metrics" / "evaluation_summary.csv"
PER_SOURCE_PATH = RESULTS_DIR / "metrics" / "per_source_precision.csv"
PREDICTIONS_PATH = RESULTS_DIR / "predictions" / "topk_predictions.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from graphrag.evaluation import evaluate_rankings

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
def load_csv_if_ready(path: Path):
    if not path.exists():
        return None
    df = pd.read_csv(path)
    if df.empty:
        return None
    return df

predictions_df = load_csv_if_ready(PREDICTIONS_PATH)
if predictions_df is None:
    raise ValueError(
        f"{PREDICTIONS_PATH} is missing or empty. Run scripts/run_experiments.py first."
    )

summary_df = load_csv_if_ready(METRICS_PATH)
per_source_df = load_csv_if_ready(PER_SOURCE_PATH)

if summary_df is None or per_source_df is None:
    inferred_k = int(predictions_df["rank"].max())
    summary_df, per_source_df = evaluate_rankings(predictions_df, k=inferred_k)

display(summary_df)


## Overall Comparison

The summary table below compares the implemented methods using mean Precision@K across evaluated source nodes.

In [ ]:
summary_sorted = summary_df.sort_values("mean_precision_at_k", ascending=False).reset_index(drop=True)
display(summary_sorted)

best_method = summary_sorted.loc[0, "method"]
best_k = int(summary_sorted.loc[0, "k"])
best_score = float(summary_sorted.loc[0, "mean_precision_at_k"])

print(f"Best method: {best_method}")
print(f"Precision@{best_k}: {best_score:.4f}")


In [ ]:
palette = ["#4c78a8", "#f58518", "#54a24b", "#e45756", "#72b7b2", "#b279a2"]
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    summary_sorted["method"],
    summary_sorted["mean_precision_at_k"],
    yerr=summary_sorted["std_precision_at_k"],
    color=palette[: len(summary_sorted)],
    capsize=6,
)
ax.set_title("Mean Precision@K by method")
ax.set_xlabel("Method")
ax.set_ylabel("Mean Precision@K")
ax.set_ylim(0, max(summary_sorted["mean_precision_at_k"].max() * 1.25, 0.05))
plt.xticks(rotation=20)
plt.tight_layout()

comparison_figure_path = FIGURES_DIR / "precision_at_k_comparison.png"
fig.savefig(comparison_figure_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved figure to: {comparison_figure_path}")


## Per-Source Performance

Averages can hide unstable behavior, so it is useful to inspect how Precision@K changes across source nodes.

In [ ]:
per_source_pivot = per_source_df.pivot(index="source", columns="method", values="precision_at_k")

fig, ax = plt.subplots(figsize=(9, 5))
per_source_pivot.boxplot(ax=ax)
ax.set_title("Per-source Precision@K distribution")
ax.set_xlabel("Method")
ax.set_ylabel("Precision@K")
plt.xticks(rotation=20)
plt.tight_layout()

per_source_figure_path = FIGURES_DIR / "per_source_precision_boxplot.png"
fig.savefig(per_source_figure_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved figure to: {per_source_figure_path}")


In [ ]:
method_hit_summary = (
    predictions_df.groupby("method", as_index=False)
    .agg(
        total_predictions=("label", "size"),
        positive_hits=("label", "sum"),
    )
)
method_hit_summary["hit_rate_within_topk"] = (
    method_hit_summary["positive_hits"] / method_hit_summary["total_predictions"]
)
display(method_hit_summary.sort_values("hit_rate_within_topk", ascending=False))


## Positive Links By Rank

This figure shows how often true positive links appear at rank 1, rank 2, and so on. Stronger ranking methods should place positive edges earlier.

In [ ]:
rank_quality = (
    predictions_df.groupby(["method", "rank"], as_index=False)["label"]
    .mean()
    .rename(columns={"label": "positive_rate"})
)

fig, ax = plt.subplots(figsize=(9, 5))
for method, subset in rank_quality.groupby("method"):
    ax.plot(subset["rank"], subset["positive_rate"], marker="o", linewidth=2, label=method)

ax.set_title("Positive edge rate by rank")
ax.set_xlabel("Rank position")
ax.set_ylabel("Fraction of positive edges")
ax.legend()
plt.tight_layout()

rank_quality_figure_path = FIGURES_DIR / "positive_rate_by_rank.png"
fig.savefig(rank_quality_figure_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved figure to: {rank_quality_figure_path}")


## Inspect Predictions From The Best Method

Use this section to show examples of the highest-ranked predicted links for the best-performing method.

In [ ]:
best_predictions = (
    predictions_df[predictions_df["method"] == best_method]
    .sort_values(["source", "rank"])
    .reset_index(drop=True)
)

display(best_predictions.head(20))

true_positive_examples = best_predictions[best_predictions["label"] == 1].head(10)
print("Example true positives retrieved by the best method")
display(true_positive_examples)


## Report Notes

You can adapt the following ideas into your report discussion:

- State which algorithm achieved the highest mean Precision@K.
- Compare local neighborhood methods against Personalized PageRank and explain why one family works better.
- Use the error bars or boxplot to discuss stability across source nodes.
- If positive links are concentrated at early ranks, explain that the method retrieves relevant context efficiently.
- Connect the result back to GraphRAG: better link prediction suggests better context retrieval for indirect relationships.